# Assignment 2: Deciphering One-Time Padded Text

**Given:** All sentences use English letters and spaces, and some sentences are padded with one or more spaces at the end. Neither punctuation nor special characters were used. Each byte of the ciphertext is written in hexadecimal format.

**Task:** Figure out the complete plaintexts.

### Thought process

- Since all sentences use English letters and spaces, without any special characters, their ASCII characters are limited to a certain range. 
- Since sentences are padded by one or more space characters at the end, they are easy to detect, just like how the question suggested.
- The given hint: write a program that identifies the possible locations of spaces, then use them to identify letters in other sentences
- If XOR-ing something gives out the space character 0X20, then it's worth investigating the end of the sentences.
- If 0X20 XOR k = c, then c XOR 0X20 = k and this key can be used in the other sentences.

### Observations on the ciphertext
- 5 out of 8 sentences end with `55`, and 4 of them end with `5E8D55` which might indicate 3 spaces in a row.
- 6 out of 8 sentences start with `F` but one hexadecimal character doesn't confirm a byte. Only two start with `F1` but that doesn't provide much additional information to me, but it can be used later. Could be uppercase letters?

### Ideas:
- Take the first sentence, go byte by byte, XOR with 0X20, take this character key and see if it works with other characters in other sentences with that index.

## Loading Ciphertext

In [ ]:
import numpy as np

# load ciphertext from file
# file contains 8 rows of ciphertext characters, each row separated by a newline
with open('ciphertexts.txt', 'r') as f:
    ciphertexts = f.read().strip().split('\n')
    
# each two characters represent a single byte in 2 hexadecimal characters
ciphertexts = [bytes.fromhex(c) for c in ciphertexts]

# convert to numpy array
ciphertexts = np.array(ciphertexts)

# define valid range of english letters and space
valid_range = list(range(0x41, 0x5B)) + list(range(0x61, 0x7B)) + [0x20]

print(ciphertexts)

[b'\xfc\xd6\xbe0\xe4\x03}\x8d\xb1\x03i\xfe\x868\xab\xfc\xd1{\xc6\xcb\xf4\xf1\xc9\xb1K\xda\xdc\xa9\xf9>\xc6HO\xf41nF.\xc9\xe8\x01\x9bu(-\x19\xcb\x97\xe2\x7f\x1a\xe7\xef\x0f{\x00\xdb\xf4\xec\x03\r\xcf\x8e\xf8\x97\x13`p\x08V\r\x17\xc2\x1b'
 b'\xe8\xc5\xae8\xe6\x07)\x9b\xe8\x04j\xf4\xc6k\xac\xf6\xc7/\xc7\xcf\xf4\xfd\x8f\xaaM\xdc\xdd\xac\xf7"\x8aP\x0c\xf21=Rk\xd8\xbe\x02\x80tm?Z\xce\x97\xe51\x1d\xa8\xfd\x14,\x08\xc7\xf2\xbe\x10\x19\xd0\x85\xbd\x95\x00mm\x01\\\r\x17\xc8\x06'
 b'\xf4\xd2\xa46\xe0\x07)\x84\xf8\x19u\xe2\x8b"\xb1\xea\xc8?\xd5\x8e\xeb\xfd\xdc\xabI\xdc\xcd\xb3\xb6#\x83GX\xa6<d\x15{\xc2\xec\x01\x8aco"\x13\xd2\x9d\xf31\x01\xb5\xbb\x13.\x1a\xc5\xfe\xfd\x0f\x03\xc9\x98\xf8\x94\x04a`\x08G\n^\x8dU'
 b'\xee\xd4\xbc<\xb2\x0bd\x98\xfe\x05j\xf0\xc5?\xff\xff\xc87\xd5\xdd\xa6\xfe\xdd\xbdY\xce\xcd\xae\xe2<\x9f\tC\xe8*r\x15}\xc9\xee\x05\x9bm|)Z\xcc\x8a\xfeg\x0b\xb4\xbb\x144I\xc5\xe5\xfb\x10\t\xd2\x9f\xf8\x83\x00{eMY\x16\r\xdeU'
 b'\xf1\xda\xa92\xb2\x12l\x9a\xe2\x18p\xf0\xc7k\xac\xfa\xd3>\xd5\xc

## Find Candidate Byte Keys
### First Byte Key Guesses
By XOR-ing the first byte of the first sentence with valid characters, candidate keys can be generated and tested across all 8 sentences. When XOR-ing with the candidate keys, they should fall within the range of the valid characters if they are truly valid.

In [20]:
# first sentence first character: XOR it with valid characters to generate candidate keys
candidate_keys = [ciphertexts[0][0] ^ a for a in valid_range]

# test candidate keys across all 8 sentences' first character
for key in candidate_keys:
    
    # decrypt the first character of each sentence
    decrypted_chars = [c[0] ^ key for c in ciphertexts]
    
    # check if all decrypted characters are valid
    if all(dc in valid_range for dc in decrypted_chars):
        print(f"Candidate key: {key}, Decrypted chars: {[chr(dc) for dc in decrypted_chars]}")

Candidate key: 189, Decrypted chars: ['A', 'U', 'I', 'S', 'L', 'C', 'L', 'M']
Candidate key: 191, Decrypted chars: ['C', 'W', 'K', 'Q', 'N', 'A', 'N', 'O']
Candidate key: 184, Decrypted chars: ['D', 'P', 'L', 'V', 'I', 'F', 'I', 'H']
Candidate key: 185, Decrypted chars: ['E', 'Q', 'M', 'W', 'H', 'G', 'H', 'I']
Candidate key: 186, Decrypted chars: ['F', 'R', 'N', 'T', 'K', 'D', 'K', 'J']
Candidate key: 187, Decrypted chars: ['G', 'S', 'O', 'U', 'J', 'E', 'J', 'K']
Candidate key: 164, Decrypted chars: ['X', 'L', 'P', 'J', 'U', 'Z', 'U', 'T']
Candidate key: 166, Decrypted chars: ['Z', 'N', 'R', 'H', 'W', 'X', 'W', 'V']
Candidate key: 157, Decrypted chars: ['a', 'u', 'i', 's', 'l', 'c', 'l', 'm']
Candidate key: 159, Decrypted chars: ['c', 'w', 'k', 'q', 'n', 'a', 'n', 'o']
Candidate key: 152, Decrypted chars: ['d', 'p', 'l', 'v', 'i', 'f', 'i', 'h']
Candidate key: 153, Decrypted chars: ['e', 'q', 'm', 'w', 'h', 'g', 'h', 'i']
Candidate key: 154, Decrypted chars: ['f', 'r', 'n', 't', 'k', '

**Observation:** Half of the candidate keys decipher them in uppercase, and the other half in lowercase. A conclusion that I could deduce is that they all either start with uppercase or lowercase letters, but it can't be determined. It truly depends on the original key that was used to encrypt, and for the decryption a key to revert it back to uppercase or lowercase won't matter at all.

An adjustment I was about to make is that I'm only going to use lowercase characters and spaces to XOR with, but check if the characters can also be uppercase. It's wrong because the original key could have been a space character, which might generate a null byte if only guessed with lowercase letters. 

Next step: Iterate over all characters across all sentences until the end, but target spaces.

In [21]:
def find_candidate_keys(i):
    
    list_of_candidate_keys = []

    # first sentence, i character: XOR it with valid characters to generate candidate keys
    candidate_keys = [ciphertexts[0][i] ^ a for a in valid_range]

    # test candidate keys across all 8 sentences' for character at position i
    for key in candidate_keys:
        
        # decrypt the i character of each sentence
        decrypted_chars = [c[i] ^ key for c in ciphertexts]
        
        # check if all decrypted characters are valid
        if all(dc in valid_range for dc in decrypted_chars):
            list_of_candidate_keys.append(key)

    return list_of_candidate_keys

i = 0

some_keys = find_candidate_keys(i)

for key in some_keys:
    decrypted_chars = [c[i] ^ key for c in ciphertexts]
    print(f"Candidate key: {key}, Decrypted chars: {[chr(dc) for dc in decrypted_chars]}")



Candidate key: 189, Decrypted chars: ['A', 'U', 'I', 'S', 'L', 'C', 'L', 'M']
Candidate key: 191, Decrypted chars: ['C', 'W', 'K', 'Q', 'N', 'A', 'N', 'O']
Candidate key: 184, Decrypted chars: ['D', 'P', 'L', 'V', 'I', 'F', 'I', 'H']
Candidate key: 185, Decrypted chars: ['E', 'Q', 'M', 'W', 'H', 'G', 'H', 'I']
Candidate key: 186, Decrypted chars: ['F', 'R', 'N', 'T', 'K', 'D', 'K', 'J']
Candidate key: 187, Decrypted chars: ['G', 'S', 'O', 'U', 'J', 'E', 'J', 'K']
Candidate key: 164, Decrypted chars: ['X', 'L', 'P', 'J', 'U', 'Z', 'U', 'T']
Candidate key: 166, Decrypted chars: ['Z', 'N', 'R', 'H', 'W', 'X', 'W', 'V']
Candidate key: 157, Decrypted chars: ['a', 'u', 'i', 's', 'l', 'c', 'l', 'm']
Candidate key: 159, Decrypted chars: ['c', 'w', 'k', 'q', 'n', 'a', 'n', 'o']
Candidate key: 152, Decrypted chars: ['d', 'p', 'l', 'v', 'i', 'f', 'i', 'h']
Candidate key: 153, Decrypted chars: ['e', 'q', 'm', 'w', 'h', 'g', 'h', 'i']
Candidate key: 154, Decrypted chars: ['f', 'r', 'n', 't', 'k', '

##  Space Detection
With the previous function, we can iterate over all characters and check for all possible keys. 

Instead of only checking if it falls withing the valid range, take the result where one of the 8 sentences contain a space. This will already fill in some of the characters in the other sentences in the respective index.

In [26]:
def find_keys_for_spaces(i):
    
    list_of_candidate_keys = []

    # first sentence, i character: XOR it with valid characters to generate candidate keys
    candidate_keys = [ciphertexts[0][i] ^ a for a in valid_range]

    # test candidate keys across all 8 sentences' for character at position i
    for key in candidate_keys:
        
        # decrypt the i character of each sentence
        decrypted_chars = [c[i] ^ key for c in ciphertexts]
        
        # check if all decrypted characters are valid and ALSO check if a space has been found in the decrypted characters
        if all(dc in valid_range for dc in decrypted_chars) and any(dc == 0x20 for dc in decrypted_chars):
            list_of_candidate_keys.append(key)

    return list_of_candidate_keys

# the decryption of the sentences, size is 8 x length of the ciphertext
decrypted_chars = np.full((len(ciphertexts), len(ciphertexts[0])), '_', dtype='<U1')

# iterate over all characters across all sentences
# save the decrypted character in position i of every sentence
# if no unique key is found, keep an underscore across all sentences
for i in range(len(ciphertexts[0])):
    keys_for_spaces = find_keys_for_spaces(i)
    
    # print(len(keys_for_spaces)) # check how many candidate keys survive
    
    if len(keys_for_spaces) == 1:
        key = keys_for_spaces[0]
        characters_for_key = [c[i] ^ key for c in ciphertexts]
        decrypted_chars[:, i] = [chr(dc) for dc in characters_for_key]
        # print(f"Candidate key: {key}, Decrypted chars: {[chr(dc) for dc in characters_for_key]}")
    else:
        decrypted_chars[:, i] = '_'


**Observations:** `print(len(keys_for_spaces))` either outputs one or zero, which means only one key can decrypt to a space character for each character position across all sentences. There are 49 positions with one surviving space candidate, and the others are none.

A partial decryption of the sentences can be made using those keys that were used to identify those spaces.

In [ ]:
# convert to list of strings for each sentence
decrypted_sentences = [''.join(chars) for chars in decrypted_chars]

print(decrypted_sentences[0])

____vate t___st_p _erifi____on across eve__ _cc_unt t_ i_cr__s_ p_o_e_tion


Some words are very obvious: They can be easily guessed.

"Activate ... verification across every account to increase protection"

I can guess that the second word is "two-step" although it was mentioned that the plaintext only contained english letters and spaces.

In [32]:
for sentence in decrypted_sentences:
    print(sentence)

____vate t___st_p _erifi____on across eve__ _cc_unt t_ i_cr__s_ p_o_e_tion
____te sys___ s_ft_are r____arly to get f__e_ f_r sof_wa_e __l_er_b_l_ties
____re lin___in_id_ mess____ sent by unre__g_iz_d or _us_ic__u_ s_n_e_s   
____ impor___t _il_s fre____tly onto sepa__t_ d_ives _o _re__n_ d_t_ _oss 
____ perso___ s_re_ns wh____er walking aw__ _o _reven_ p_ys__a_ a_c_s_    
____k app ___mi_si_ns pe____ically and re__k_ u_neces_ar_ a__e_s _i_h_s   
____t pers___l _et_ils s____d publicly on__n_ t_ safe_ua_d __g_ta_ _r_vacy
____tor em___in_ o_line ____s constantly __ _em_in co_pl_te__ _re_a_e_    


### Guessing The Missing Letters

From second sentence, the second word seems like "system". We can use this to guess the key for all sentences instead.

My guess for all sentences:

- `Activate two-step verification across every account to increase protection`
- `****te system software regularly to get f**e* for software vulnerabilities` (Update, fixes)
- `****re lin***in*id messages sent by unrecognized or suspicious senders`
- `Backup important files frequently onto separate drives to prevent data loss`
- `**** personal screens whenever walking away to prevent physical access` (Hide)
- `****k app permissions periodically and revoke unnecessary access *i*h*s` (Check)
- `****t personal details s****d publicly online to safeguard **g*ta* privacy`(sended)
- `****tor em***in* online ****s constantly to remain completely *re*a*e*` 

I think I could deduce what the second sentence is, so I'm using this guess to recover the key to use on all sentences. The decrypted sentences should make sense.

`M = "Update system software regularly to get fixes for software vulnerabilities"`

Since 

$$ C = M \oplus K $$

Then

$$ K = C \oplus M $$

where M is the plaintext, and C is the given ciphertext. After acquiring the key, It will be used on all sentences to decipher them.

In [33]:
M = "Update system software regularly to get fixes for software vulnerabilities"

K = bytes([c ^ ord(m) for c, m in zip(ciphertexts[1], M)])

# Use K to decrypt all ciphertexts
decrypted_sentences_with_K = [''.join(chr(c ^ k) for c, k in zip(ciphertext, K)) for ciphertext in ciphertexts]

for sentence in decrypted_sentences_with_K:
    print(sentence)

Activate two-step verification across every account to increase protection
Update system software regularly to get fixes for software vulnerabilities
Ignore links inside messages sent by unrecognized or suspicious senders   
Save important files frequently onto separate drives to prevent data loss 
Lock personal screens whenever walking away to prevent physical access    
Check app permissions periodically and revoke unnecessary access rights   
Limit personal details shared publicly online to safeguard digital privacy
Monitor emerging online scams constantly to remain completely prepared    
